# Pixel Mappers

Example of custom mapping.

In [1]:
from dataclasses import dataclass

import numpy as np

from ledmap.pixels import Mapper

## Default Mapper

In [2]:
Mapper((64,)).to_mapping()

64 pixels 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63

In [3]:
Mapper((3, 7)).to_mapping()

0,1,2,3,4,5,6
3,4,5,6,7,8,9
6,7,8,9,10,11,12


## Fairy/seed pixel curtain

Similar to these: https://www.youtube.com/watch?v=URwnwcRwbFo


In [4]:
@dataclass
class Curtain(Mapper):
    """Fairy/seed pixel curtain."""

    delta: int

    def get(self, y: int, x: int) -> int:
        """Map pixel location to index."""
        return x * self.delta + y

    def check(self) -> None:
        """Check mapper parameters."""
        super().check()
        msg = None
        if len(self.shape) != 2:
            msg = "Array must be 2D."
        elif self.delta < self.shape[0]:
            msg = "Delta must be at least matrix height."
        if msg:
            raise ValueError(msg)

In [5]:
Curtain((5, 5), 10).to_mapping()

0,10,20,30,40
1,11,21,31,41
2,12,22,32,42
3,13,23,33,43
4,14,24,34,44


In [6]:
@dataclass
class ExtendedCurtain(Mapper):
    """Fairy/seed pixel curtain including header.

    Header is where the signal is split for each column. This can be addressed,
    but is oddly-referenced.
    """

    delta: int

    def get(self, y: int, x: int) -> int:
        """Map pixel location to index."""
        return (x + (y <= 0)) * self.delta + y - 1

    def check(self) -> None:
        """Check mapper parameters."""
        super().check()
        msg = None
        if len(self.shape) != 2:
            msg = "Array must be 2D."
        elif self.delta < self.shape[0]:
            msg = "Delta must be at least matrix height."
        if msg:
            raise ValueError(msg)

In [7]:
ExtendedCurtain((5, 4), 10).to_mapping()

9,19,29,39
0,10,20,30
1,11,21,31
2,12,22,32
3,13,23,33


## An actual curtain

My hard-coded ESPHome config (including header) contains:

```yaml
  width: 20
  height: 21
  pixel_mapper: |-
    return 799 - 40 * (x + (y >= 0)) + y;
```

In [8]:
m = ExtendedCurtain((21, 20), 40).to_mapping()
m = m.apply(np.fliplr).apply(lambda x: np.where(x <= 798, x, -1))
m

-1,759,719,679,639,599,559,519,479,439,399,359,319,279,239,199,159,119,79,39
760,720,680,640,600,560,520,480,440,400,360,320,280,240,200,160,120,80,40,0
761,721,681,641,601,561,521,481,441,401,361,321,281,241,201,161,121,81,41,1
762,722,682,642,602,562,522,482,442,402,362,322,282,242,202,162,122,82,42,2
763,723,683,643,603,563,523,483,443,403,363,323,283,243,203,163,123,83,43,3
764,724,684,644,604,564,524,484,444,404,364,324,284,244,204,164,124,84,44,4
765,725,685,645,605,565,525,485,445,405,365,325,285,245,205,165,125,85,45,5
766,726,686,646,606,566,526,486,446,406,366,326,286,246,206,166,126,86,46,6
767,727,687,647,607,567,527,487,447,407,367,327,287,247,207,167,127,87,47,7
768,728,688,648,608,568,528,488,448,408,368,328,288,248,208,168,128,88,48,8
769,729,689,649,609,569,529,489,449,409,369,329,289,249,209,169,129,89,49,9


In [9]:
from ledmap.util import get_string

print(get_string(m.dump_wled))

{"map":[-1,759,719,679,639,599,559,519,479,439,399,359,319,279,239,199,159,119,79,39,760,720,680,640,600,560,520,480,440,400,360,320,280,240,200,160,120,80,40,0,761,721,681,641,601,561,521,481,441,401,361,321,281,241,201,161,121,81,41,1,762,722,682,642,602,562,522,482,442,402,362,322,282,242,202,162,122,82,42,2,763,723,683,643,603,563,523,483,443,403,363,323,283,243,203,163,123,83,43,3,764,724,684,644,604,564,524,484,444,404,364,324,284,244,204,164,124,84,44,4,765,725,685,645,605,565,525,485,445,405,365,325,285,245,205,165,125,85,45,5,766,726,686,646,606,566,526,486,446,406,366,326,286,246,206,166,126,86,46,6,767,727,687,647,607,567,527,487,447,407,367,327,287,247,207,167,127,87,47,7,768,728,688,648,608,568,528,488,448,408,368,328,288,248,208,168,128,88,48,8,769,729,689,649,609,569,529,489,449,409,369,329,289,249,209,169,129,89,49,9,770,730,690,650,610,570,530,490,450,410,370,330,290,250,210,170,130,90,50,10,771,731,691,651,611,571,531,491,451,411,371,331,291,251,211,171,131,91,51,11,7